## Chargement des Données via SQLAlchemy

Configurer la connexion SQLAlchemy

In [10]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

user = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")
host = os.getenv("DB_HOST")
port = os.getenv("DB_PORT")
db = os.getenv("DB_NAME")

if not all([user, password, db]):
    raise ValueError("One or more DB environment variables are missing! Check your .env file.")

# connection string
engine = create_engine(f"postgresql://{user}:{password}@{host}:{port}/{db}")

# test connection
try:
    with engine.connect() as conn:
        print('Connection successful!')
except Exception as e:
    print(f'connection failed: {e}')

Connection successful!


Séparer financecore_clean.csv selon les tables normalisées avant insertion


In [11]:
import pandas as pd
df = pd.read_csv("../data/financecore_clean.csv")

print(df.head())
print(df.columns)

  transaction_id client_id     date_transaction  annee  mois  trimestre  \
0      TXN000559   CLI0060  2022-04-19 02:31:00   2022     4          2   
1      TXN001154   CLI0057  2024-06-20 20:51:00   2024     6          2   
2      TXN000764   CLI0015  2024-08-28 05:03:00   2024     8          3   
3      TXN001598   CLI0045  2024-01-07 08:16:00   2024     1          1   
4      TXN001873   CLI0034  2024-08-11 19:52:00   2024     8          3   

  jour_semaine      categorie              produit type_operation  ...  \
0      Tuesday  Depot especes       Compte Epargne         Credit  ...   
1     Thursday    Retrait DAB  Credit Consommation          Debit  ...   
2    Wednesday    Prelevement                  PEA          Debit  ...   
3       Sunday    Paiement CB  Credit Consommation         Credit  ...   
4       Sunday       Interets    Credit Immobilier         Credit  ...   

   montant devise  taux_change_eur montant_eur  montant_eur_verifie  \
0  2050.42    EUR             1.0

In [12]:
df['date_transaction'] = pd.to_datetime(df['date_transaction'])

# 2. Dimensions
agencies_df = df[['agence']].drop_duplicates().reset_index(drop=True)
products_df = df[['produit']].drop_duplicates().reset_index(drop=True)
categories_df = df[['categorie']].drop_duplicates().reset_index(drop=True)
clients_df = df[['client_id', 'score_credit_client', 'categorie_risque', 'segment_client']].drop_duplicates(subset=['client_id'])

# 3. Time Dimension (Matching your 'dim_date' table)
dim_date = pd.DataFrame({'date_transaction': df['date_transaction'].unique()})
dim_date['annee'] = dim_date['date_transaction'].dt.year
dim_date['mois'] = dim_date['date_transaction'].dt.month
dim_date['trimestre'] = dim_date['date_transaction'].dt.quarter
dim_date['jour_semaine'] = dim_date['date_transaction'].dt.day_name()

print("Step 'Séparer financecore_clean.csv' is complete!")
print(f'- {len(agencies_df)} Unique agencies')
print(f'- {len(products_df)} Unique products')
print(f'- {len(categories_df)} Unique categories')
print(f'- {len(clients_df)} Unique clients')

Step 'Séparer financecore_clean.csv' is complete!
- 9 Unique agencies
- 8 Unique products
- 8 Unique categories
- 150 Unique clients


Insérer les données

In [13]:
# A. Safe insertion function
def safe_insert(dataframe, table_name, engine):
    try:
        dataframe.to_sql(table_name, engine, if_exists='append', index=False)
        print(f"✅ {table_name}: Inserted.")
    except Exception:
        print(f"ℹ️ {table_name}: Already contains data, skipping.")

# B. Insert Dimensions
safe_insert(agencies_df, 'agencies', engine)
safe_insert(products_df, 'products', engine)
safe_insert(categories_df, 'categories', engine)
safe_insert(clients_df, 'clients', engine)
safe_insert(dim_date, 'dim_date', engine)

# C. Map IDs for the Transactions Table
agencies_db = pd.read_sql("SELECT agence_id, agence FROM agencies", engine)
products_db = pd.read_sql("SELECT produit_id, produit FROM products", engine)
categories_db = pd.read_sql("SELECT categorie_id, categorie FROM categories", engine)
dates_db = pd.read_sql("SELECT date_id, date_transaction FROM dim_date", engine)

# D. Merge logic
df_mapped = df.merge(agencies_db, on='agence') \
              .merge(products_db, on='produit') \
              .merge(categories_db, on='categorie') \
              .merge(dates_db, on='date_transaction')

# E. Select exactly the 14 columns defined in your 'create_tables.sql'
transactions_df = df_mapped[[
    'transaction_id', 'client_id', 'agence_id', 'produit_id', 'categorie_id', 
    'date_id', 'montant', 'type_operation', 'devise', 'taux_change_eur', 
    'montant_eur_verifie', 'statut', 'is_anomaly', 'solde_avant'
]]

# F. Final Insertion
try:
    transactions_df.to_sql('transactions', engine, if_exists='append', index=False)
    print(f"🚀 SUCCESS: {len(transactions_df)} transactions inserted!")
except Exception as e:
    print(f"❌ Final insertion failed: {e}")

✅ agencies: Inserted.
✅ products: Inserted.
✅ categories: Inserted.
✅ clients: Inserted.
✅ dim_date: Inserted.
🚀 SUCCESS: 2000 transactions inserted!


Implémenter la gestion des conflits (ON CONFLICT DO NOTHING / DO UPDATE)

In [14]:
from sqlalchemy import text

# 1. DE-DUPLICATE in Python first (Crucial step to avoid CardinalityViolation)
# This keeps only the LATEST version of each transaction ID in your current batch
clean_batch_df = transactions_df.drop_duplicates(subset=['transaction_id'], keep='last')

# 2. Upload the clean batch to a temporary table
clean_batch_df.to_sql('temp_transactions', engine, if_exists='replace', index=False)

# 3. Execute the UPSERT
upsert_sql = text("""
INSERT INTO transactions (
    transaction_id, client_id, agence_id, produit_id, categorie_id, 
    date_id, montant, type_operation, devise, taux_change_eur, 
    montant_eur_verifie, statut, is_anomaly, solde_avant
)
SELECT * FROM temp_transactions
ON CONFLICT (transaction_id) 
DO UPDATE SET 
    statut = EXCLUDED.statut,
    is_anomaly = EXCLUDED.is_anomaly,
    solde_avant = EXCLUDED.solde_avant,
    montant = EXCLUDED.montant,
    montant_eur_verifie = EXCLUDED.montant_eur_verifie;

-- Clean up
DROP TABLE temp_transactions;
""")

with engine.begin() as conn:
    conn.execute(upsert_sql)
    print(f"✅ Successfully processed {len(clean_batch_df)} unique transactions.")

✅ Successfully processed 2000 unique transactions.


Vérifier l'intégrité référentielle après chargement (COUNT…..)

In [15]:
# 1. Ensure Python count only includes UNIQUE transaction IDs
# This removes the duplicates causing the '4000' count in your screenshot
final_unique_df = transactions_df.drop_duplicates(subset=['transaction_id'])

# 2. Re-run the Integrity Check
csv_count = len(final_unique_df)
db_count = pd.read_sql("SELECT COUNT(*) FROM transactions", engine).iloc[0,0]

print("--- Final Sync Report ---")
print(f"Unique Transactions (Python): {csv_count}")
print(f"Total Transactions (Database): {db_count}")

if db_count == csv_count:
    print("✅ Perfect Sync: Database matches unique CSV records.")
else:
    print(f"ℹ️ Total records in DB: {db_count}. Your current batch has {csv_count} unique rows.")

--- Final Sync Report ---
Unique Transactions (Python): 2000
Total Transactions (Database): 2000
✅ Perfect Sync: Database matches unique CSV records.


## Requêtes SQL Analytiques Avancées

Agrégations : total et moyenne des transactions par agence, produit et mois (GROUP BY, HAVING)

In [16]:
# 1. Total and Average transactions by Agency, Product, and Month
aggregation_query = """
SELECT 
    a.agence,
    p.produit,
    d.mois,
    d.annee,
    SUM(t.montant_eur_verifie) AS total_montant_eur,
    ROUND(AVG(t.montant_eur_verifie)::numeric, 2) AS moyenne_montant_eur,
    COUNT(t.transaction_id) AS nombre_transactions
FROM transactions t
JOIN agencies a ON t.agence_id = a.agence_id
JOIN products p ON t.produit_id = p.produit_id
JOIN dim_date d ON t.date_id = d.date_id
GROUP BY a.agence, p.produit, d.annee, d.mois
HAVING COUNT(t.transaction_id) > 1 -- Filters for statistical significance
ORDER BY d.annee DESC, d.mois DESC, total_montant_eur DESC;
"""

df_aggr = pd.read_sql(aggregation_query, engine)
df_aggr.head(10)

,agence,produit,mois,annee,total_montant_eur,moyenne_montant_eur,nombre_transactions
0,Toulouse-Capitole,Credit Immobilier,12,2024,4031.42,2015.71,2
1,Lille-Grand-Place,Compte Courant,12,2024,2300.01,1150.01,2
2,Lyon-Part-Dieu,Credit Immobilier,12,2024,1836.79,918.40,2
3,Paris-Centre,Credit Auto,12,2024,1782.68,891.34,2
4,Paris-Centre,Credit Consommation,12,2024,356.54,178.27,2
5,Lyon-Part-Dieu,Compte Courant,12,2024,210.11,105.06,2
6,Inconnue,PEA,12,2024,190.09,95.05,2
7,Nantes-Commerce,PEA,12,2024,-231.85,-115.93,2
8,Lille-Grand-Place,Livret A,12,2024,-550.01,-275.01,2
9,Bordeaux-Meriadeck,Credit Auto,12,2024,-987.67,-493.84,2


Sous-requêtes : clients avec un solde inférieur à la moyenne nationale

In [17]:
subquery_logic = """
SELECT
    c.client_id,
    c.segment_client,
    c.categorie_risque,
    t.solde_avant
FROM clients c
JOIN transactions t ON c.client_id = t.client_id
WHERE t.solde_avant < (
    -- this subquery calculates the national average
    SELECT AVG (solde_avant)
    FROM transactions
)
-- grouping to ensure we see the unique clients meeting the criteria
GROUP BY c.client_id, c.segment_client, c.categorie_risque, t.solde_avant
ORDER BY t.solde_avant ASC;
"""

df_low_balance = pd.read_sql(subquery_logic, engine)
df_low_balance.head(10)

,client_id,segment_client,categorie_risque,solde_avant
0,CLI0082,Premium,Low,103.04
1,CLI0003,Standard,Medium,112.31
2,CLI0016,Standard,Low,127.18
3,CLI0101,Standard,Medium,179.12
4,CLI0149,Risque,High,212.30
5,CLI0075,Standard,Medium,222.09
6,CLI0072,Standard,Medium,260.32
7,CLI0133,Standard,Medium,267.60
8,CLI0117,Standard,Medium,308.61
9,CLI0118,Standard,Medium,335.20


CASE WHEN : calcul du taux de défaut par segment de risque

In [18]:
risk_analysis_query = """
SELECT 
    c.categorie_risque,
    COUNT(t.transaction_id) as total_transaction,
    SUM(CASE WHEN t.statut = 'Rejete' THEN 1 ELSE 0 END) as nb_rejets,
    ROUND(
        (SUM(CASE WHEN t.statur = 'Rejete') THEN 1 ELSE 0 END)::numeric /
        COUNT(t.transaction_id)::numeric) * 100, 2
        

_IncompleteInputError: incomplete input (2963649380.py, line 1)

Jointures multi-tables

Créer des vues analytiques pour les KPIs du dashboard (brief 3)